# BreastMNIST Study

## Setup and Reproducibility

!pip install -q "numpy==1.26.4" "scipy==1.13.1"

!pip install -q "transformers==4.44.2" --no-deps

!pip install -q "tokenizers>=0.19,<0.20" "huggingface-hub>=0.23,<0.25" "safetensors>=0.4"

!pip install -q "medmnist" "tf-keras"

In [1]:
# SETUP AND REPRODUCIBILITY 

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['PYTHONHASHSEED'] = '42'

import random
import numpy as np
import tensorflow as tf

# Global reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# Imports
import medmnist
from medmnist import INFO
import gc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models, applications, mixed_precision, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from transformers import TFViTForImageClassification
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              roc_auc_score, precision_score, recall_score, f1_score,
                              average_precision_score, log_loss)
from sklearn.calibration import calibration_curve
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

print(f"NumPy: {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Random Seed: {SEED}")
print(f"Legacy Keras: {os.environ.get('TF_USE_LEGACY_KERAS')}")

2026-06-12 04:10:35.468465: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781237435.490539    3540 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781237435.497722    3540 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781237435.515676    3540 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781237435.515694    3540 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781237435.515696    3540 computation_placer.cc:177] computation placer alr

NumPy: 1.26.4
TensorFlow: 2.19.0
Random Seed: 42
Legacy Keras: 1


## Configuration

In [2]:
# CONFIGURATION

DATASET = 'breastmnist'
IMAGE_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 30
LR_VIT = 2e-5
LR_CNN = 1e-4
DROPOUT_META = 0.5
META_BATCH_SIZE = 16
META_EPOCHS = 50
META_PATIENCE = 5
PATIENCE_EARLY = 7
PATIENCE_REDUCE = 3
REDUCE_FACTOR = 0.5
MIN_LR = 1e-6

OUTPUT_DIR = f"/kaggle/working/{DATASET}_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mixed precision for speed
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(f"Mixed Precision: {policy.name}")
print(f"Output Dir: {OUTPUT_DIR}")

Your GPU may run slowly with dtype policy mixed_float16 because it does not have compute capability of at least 7.0. Your GPU:
  Tesla P100-PCIE-16GB, compute capability 6.0
See https://developer.nvidia.com/cuda-gpus for a list of GPUs and their compute capabilities.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
Mixed Precision: mixed_float16
Output Dir: /kaggle/working/breastmnist_outputs


## Data Loading

In [3]:
# DATA LOADER

def load_medmnist_data(dataset_flag, size=224, batch_size=8):
    """Loads MedMNIST v2 dataset and prepares tf.data pipelines."""
    info = INFO[dataset_flag]
    n_classes = len(info['label'])
    DataClass = getattr(medmnist, info['python_class'])

    print(f"Loading {dataset_flag}...")
    train_data = DataClass(split='train', download=True, size=size)
    val_data = DataClass(split='val', download=True, size=size)
    test_data = DataClass(split='test', download=True, size=size)

    X_train, y_train = train_data.imgs, train_data.labels
    X_val, y_val = val_data.imgs, val_data.labels
    X_test, y_test = test_data.imgs, test_data.labels

    # Convert grayscale to RGB if needed
    if X_train.shape[-1] != 3:
        X_train = np.repeat(X_train[..., np.newaxis], 3, -1)
        X_val = np.repeat(X_val[..., np.newaxis], 3, -1)
        X_test = np.repeat(X_test[..., np.newaxis], 3, -1)

    X_train = X_train.astype('float32')
    X_val = X_val.astype('float32')
    X_test = X_test.astype('float32')

    y_train_enc = to_categorical(y_train, n_classes)
    y_val_enc = to_categorical(y_val, n_classes)
    y_test_enc = to_categorical(y_test, n_classes)

    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train_enc))\
        .shuffle(1000, seed=SEED).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val_enc)).batch(batch_size)
    test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test_enc)).batch(batch_size)

    return train_ds, val_ds, test_ds, n_classes, y_test, X_test, info

train_ds, val_ds, test_ds, n_classes, y_test_true, X_test_raw, dataset_info = \
    load_medmnist_data(DATASET, size=IMAGE_SIZE, batch_size=BATCH_SIZE)
y_true_flat = y_test_true.flatten()
y_true_cat = to_categorical(y_true_flat, n_classes)
CLASS_NAMES = [dataset_info['label'][str(i)] for i in range(n_classes)]
print(f"Classes: {n_classes}")
print(f"Class Names: {CLASS_NAMES}")
print(f"Test set size: {len(y_true_flat)}")

Loading breastmnist...


I0000 00:00:1781237446.671266    3540 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Classes: 2
Class Names: ['malignant', 'normal, benign']
Test set size: 156


## Class Distribution Analysis

In [4]:
# CLASS DISTRIBUTION ANALYSIS

DataClass = getattr(medmnist, dataset_info['python_class'])
train_labels = DataClass(split='train', size=IMAGE_SIZE).labels.flatten()
val_labels = DataClass(split='val', size=IMAGE_SIZE).labels.flatten()
test_labels = DataClass(split='test', size=IMAGE_SIZE).labels.flatten()

dist_rows = []
for i, name in enumerate(CLASS_NAMES):
    train_count = (train_labels == i).sum()
    val_count = (val_labels == i).sum()
    test_count = (test_labels == i).sum()
    total = train_count + val_count + test_count
    dist_rows.append({
        'Class ID': i,
        'Class Name': name,
        'Train': train_count,
        'Val': val_count,
        'Test': test_count,
        'Total': total,
        '% of Total': round(100 * total / (len(train_labels)+len(val_labels)+len(test_labels)), 2)
    })

dist_df = pd.DataFrame(dist_rows)
print("\n=== CLASS DISTRIBUTION TABLE ===")
print(dist_df.to_string(index=False))

ir = dist_df['Train'].max() / dist_df['Train'].min()
print(f"\nImbalance Ratio (IR): {ir:.2f}")
print(f"Total samples: Train={len(train_labels)}, Val={len(val_labels)}, Test={len(test_labels)}")
print(f"Missing values check: Train NaNs={pd.isna(train_labels).sum()}, Test NaNs={pd.isna(test_labels).sum()}")
print(f"Duplicates: handled by MedMNIST v2 official splits (no overlap between train/val/test)")

dist_df.to_csv(f"{OUTPUT_DIR}/class_distribution.csv", index=False)


=== CLASS DISTRIBUTION TABLE ===
 Class ID     Class Name  Train  Val  Test  Total  % of Total
        0      malignant    147   21    42    210       26.92
        1 normal, benign    399   57   114    570       73.08

Imbalance Ratio (IR): 2.71
Total samples: Train=546, Val=78, Test=156
Missing values check: Train NaNs=0, Test NaNs=0
Duplicates: handled by MedMNIST v2 official splits (no overlap between train/val/test)


## Model Factory Function

In [5]:
# MODEL FACTORY

def get_model(model_name, n_classes, input_shape=(224, 224, 3)):
    """Builds a model with the specified architecture."""
    inputs = layers.Input(shape=input_shape)

    if model_name == 'EfficientNetV2M':
        base = applications.EfficientNetV2M(include_top=False, weights='imagenet', input_tensor=inputs)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'InceptionResNetV2':
        x = layers.Rescaling(1./127.5, offset=-1)(inputs)
        base = applications.InceptionResNetV2(include_top=False, weights='imagenet', input_tensor=x)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'ConvNeXtBase':
        base = applications.ConvNeXtBase(include_top=False, weights='imagenet', input_tensor=inputs)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'ViT-Base':
        x = layers.Rescaling(1./127.5, offset=-1)(inputs)
        x = layers.Permute((3, 1, 2))(x)
        backbone = TFViTForImageClassification.from_pretrained(
            'google/vit-base-patch16-224', num_labels=n_classes,
            ignore_mismatched_sizes=True, use_safetensors=False)
        outputs = backbone.vit(x)[0]
        outputs = backbone.classifier(outputs[:, 0, :])
        outputs = layers.Activation('softmax', dtype='float32')(outputs)

    else:
        raise ValueError(f"Unknown Model: {model_name}")

    return models.Model(inputs=inputs, outputs=outputs, name=model_name)

## Trainer Function

In [6]:
# TRAINER

def train_and_save(model_name):
    print(f"\n{'='*20} TRAINING {model_name} {'='*20}")
    tf.keras.backend.clear_session()
    gc.collect()

    model_path = f"{OUTPUT_DIR}/{model_name}_{DATASET}.keras"
    val_pred_path = f"{OUTPUT_DIR}/{model_name}_val_preds.npy"
    test_pred_path = f"{OUTPUT_DIR}/{model_name}_test_preds.npy"

    if os.path.exists(model_path):
        print(f"Loading existing model: {model_path}")
        model = get_model(model_name, n_classes)
        model.load_weights(model_path)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    else:
        model = get_model(model_name, n_classes)
        lr = LR_VIT if model_name == 'ViT-Base' else LR_CNN
        model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        cb = [
            ModelCheckpoint(model_path, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=REDUCE_FACTOR, patience=PATIENCE_REDUCE, min_lr=MIN_LR, verbose=1),
            EarlyStopping(monitor='val_accuracy', patience=PATIENCE_EARLY, restore_best_weights=True, verbose=1)
        ]
        model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cb, verbose=1)
        model.load_weights(model_path)

    # Save predictions
    print(f"Generating predictions...")
    val_preds = model.predict(val_ds, verbose=0)
    test_preds = model.predict(test_ds, verbose=0)
    np.save(val_pred_path, val_preds)
    np.save(test_pred_path, test_preds)

    acc = accuracy_score(y_true_flat, np.argmax(test_preds, axis=1))
    print(f"{model_name} Test Accuracy: {acc:.4f}")

    del model
    gc.collect()
    return acc

## Model Training

In [7]:
train_and_save('EfficientNetV2M')


==================== TRAINING EfficientNetV2M ====================
Epoch 1/30


E0000 00:00:1781237538.270868    3540 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inEfficientNetV2M/block1b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1781237547.032901    3594 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1781237555.190095    3593 service.cc:152] XLA service 0x7c3f0c002f60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781237555.190136    3593 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1781237555.488800    3594 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


137/137 [==============================] - ETA: 0s - loss: 0.5341 - accuracy: 0.7271
Epoch 1: val_accuracy improved from -inf to 0.91026, saving model to /kaggle/working/breastmnist_outputs/EfficientNetV2M_breastmnist.keras
137/137 [==============================] - 282s 882ms/step - loss: 0.5341 - accuracy: 0.7271 - val_loss: 0.2872 - val_accuracy: 0.9103 - lr: 1.0000e-04
Epoch 2/30
137/137 [==============================] - ETA: 0s - loss: 0.3185 - accuracy: 0.8590
Epoch 2: val_accuracy did not improve from 0.91026
137/137 [==============================] - 102s 742ms/step - loss: 0.3185 - accuracy: 0.8590 - val_loss: 0.2574 - val_accuracy: 0.8846 - lr: 1.0000e-04
Epoch 3/30
137/137 [==============================] - ETA: 0s - loss: 0.2010 - accuracy: 0.9249
Epoch 3: val_accuracy did not improve from 0.91026
137/137 [==============================] - 102s 741ms/step - loss: 0.2010 - accuracy: 0.9249 - val_loss: 0.2508 - val_accuracy: 0.9103 - lr: 1.0000e-04
Epoch 4/30
137/137 [======

0.8717948717948718

In [8]:
train_and_save('ConvNeXtBase')


==================== TRAINING ConvNeXtBase ====================
350926856/350926856 [==============================] - 1s 0us/step
Epoch 1/30


2026-06-12 04:36:13.031715: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-12 04:36:13.216997: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


137/137 [==============================] - ETA: 0s - loss: 0.6231 - accuracy: 0.7326
Epoch 1: val_accuracy improved from -inf to 0.60256, saving model to /kaggle/working/breastmnist_outputs/ConvNeXtBase_breastmnist.keras
137/137 [==============================] - 366s 2s/step - loss: 0.6231 - accuracy: 0.7326 - val_loss: 0.6057 - val_accuracy: 0.6026 - lr: 1.0000e-04
Epoch 2/30
137/137 [==============================] - ETA: 0s - loss: 0.3669 - accuracy: 0.8535
Epoch 2: val_accuracy improved from 0.60256 to 0.89744, saving model to /kaggle/working/breastmnist_outputs/ConvNeXtBase_breastmnist.keras
137/137 [==============================] - 48s 347ms/step - loss: 0.3669 - accuracy: 0.8535 - val_loss: 0.3558 - val_accuracy: 0.8974 - lr: 1.0000e-04
Epoch 3/30
137/137 [==============================] - ETA: 0s - loss: 0.1701 - accuracy: 0.9432
Epoch 3: val_accuracy improved from 0.89744 to 0.96154, saving model to /kaggle/working/breastmnist_outputs/ConvNeXtBase_breastmnist.keras
137/137 [

0.9102564102564102

In [9]:
train_and_save('ViT-Base')


==================== TRAINING ViT-Base ====================


config.json: 0.00B [00:00, ?B/s]

tf_model.h5:   0%|          | 0.00/347M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFViTForImageClassification.

Some weights of TFViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier/kernel:0: found shape (768, 1000) in the checkpoint and (768, 2) in the model instantiated
- classifier/bias:0: found shape (1000,) in the checkpoint and (2,) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/30
137/137 [==============================] - ETA: 0s - loss: 0.3920 - accuracy: 0.8388
Epoch 1: val_accuracy improved from -inf to 0.91026, saving model to /kaggle/working/breastmnist_outputs/ViT-Base_breastmnist.keras
137/137 [==============================] - 71s 204ms/step - loss: 0.3920 - accuracy: 0.8388 - val_loss: 0.2670 - val_accuracy: 0.9103 - lr: 2.0000e-05
Epoch 2/30
137/137 [==============================] - ETA: 0s - loss: 0.1629 - accuracy: 0.9487
Epoch 2: val_accuracy did not improve from 0.91026
137/137 [==============================] - 19s 140ms/step - loss: 0.1629 - accuracy: 0.9487 - val_loss: 0.3346 - val_accuracy: 0.8590 - lr: 2.0000e-05
Epoch 3/30
137/137 [==============================] - ETA: 0s - loss: 0.0682 - accuracy: 0.9817
Epoch 3: val_accuracy improved from 0.91026 to 0.92308, saving model to /kaggle/working/breastmnist_outputs/ViT-Base_breastmnist.keras
137/137 [==============================] - 26s 190ms/step - loss: 0.0682 - accuracy: 0.9817 

0.9166666666666666

In [10]:
train_and_save('InceptionResNetV2')


==================== TRAINING InceptionResNetV2 ====================
219055592/219055592 [==============================] - 1s 0us/step
Epoch 1/30
137/137 [==============================] - ETA: 0s - loss: 0.5297 - accuracy: 0.7454
Epoch 1: val_accuracy improved from -inf to 0.73077, saving model to /kaggle/working/breastmnist_outputs/InceptionResNetV2_breastmnist.keras
137/137 [==============================] - 142s 611ms/step - loss: 0.5297 - accuracy: 0.7454 - val_loss: 0.5504 - val_accuracy: 0.7308 - lr: 1.0000e-04
Epoch 2/30
137/137 [==============================] - ETA: 0s - loss: 0.3982 - accuracy: 0.8333
Epoch 2: val_accuracy improved from 0.73077 to 0.83333, saving model to /kaggle/working/breastmnist_outputs/InceptionResNetV2_breastmnist.keras
137/137 [==============================] - 33s 240ms/step - loss: 0.3982 - accuracy: 0.8333 - val_loss: 0.4050 - val_accuracy: 0.8333 - lr: 1.0000e-04
Epoch 3/30
137/137 [==============================] - ETA: 0s - loss: 0.2597 - accu

0.8910256410256411